# 01 - SQL 基础（SQL Fundamentals）
> Harvard CS50’s Intro to Databases with SQL  |  课程时间：00:04 – 1:06:22

这个 notebook 带你一步步上手 SQL。**不需要安装任何东西**—— Python 内置的 `sqlite3` 就是你的数据库引擎。

## 怎么用？
1. 从上到下依次运行每个 cell（`Shift + Enter`）
2. 观察 SQL 的执行结果
3. 在每个 ✏️ 标记处自己写 SQL 动手试

## 今日学习路线

| # | 内容 | 对应视频 |
|---|------|----------|
| 1.1 | 数据库简介 | 00:04 |
| 1.2 | 环境配置 — 创建数据库 | 13:36 |
| 1.3 | 基本查询 — SELECT, WHERE, LIMIT | 21:51 |
| 1.4 | 过滤与运算符 — LIKE, NULL, AND/OR/NOT | 32:18 |
| 1.5 | 范围与模式匹配 — BETWEEN, IN | 46:59 |
| 1.6 | 排序 — ORDER BY | 57:53 |
| 1.7 | 聚合函数 — COUNT, AVG, MAX, MIN, SUM | 1:06:22 |

---
## 1.1 数据库简介

### 为什么不用 Excel？

| | 电子表格 | 数据库 |
|---|----------|--------|
| 数据量 | 几万行就卡 | 百万~数十亿行 |
| 多人协作 | 容易冲突 | 支持并发访问 |
| 数据冗余 | 严重（同一作者重复 1000 次） | 通过多表设计消除 |
| 查询能力 | 筛选+排序 | 任意复杂的 SQL 查询 |
| 数据安全 | 删了就是删了 | 事务保护、权限控制 |

### SQL 是什么？

**S**tructured **Q**uery **L**anguage — 结构化查询语言。操作关系型数据库的统一语言。

```
DQL (查询):  SELECT              ← 这章主要学这个
DML (操作):  INSERT, UPDATE, DELETE
DDL (定义):  CREATE, ALTER, DROP
DCL (控制):  GRANT, REVOKE
```

### 我们用的数据

一个模拟的**纽约时报畅销书榜单**数据库（类似 CS50 的 `longlist`），包含 25 本书的信息。

---
## 1.2 环境配置 — 创建数据库 & 写入数据

> ⚠️ **首先运行这个 cell！** 它会创建数据库和一张 `books` 表，后面所有查询都依赖它。

In [ ]:
import sqlite3

# ============================================
# 辅助函数：执行 SQL 并美化输出
# 后面所有 cell 都用 sql('...') 来查询
# ============================================
def sql(query):
    """执行 SQL 查询并打印结果表格。query 可以包含多条语句（用 ; 分隔），
       只有最后一条 SELECT 的结果会打印。"""
    statements = [s.strip() for s in query.strip().split(';') if s.strip()]
    if not statements:
        return
    
    result = None
    for stmt in statements:
        cursor = conn.execute(stmt)
        # 只保留最后一条有返回结果的语句
        if cursor.description:
            result = cursor.fetchall()
            columns = [d[0] for d in cursor.description]
    
    conn.commit()
    
    if result is None:
        print('✅ Done')
        return
    
    if not result:
        print('(empty — 0 rows)')
        return
    
    # 计算列宽
    col_widths = [len(c) for c in columns]
    for row in result:
        for i, val in enumerate(row):
            col_widths[i] = max(col_widths[i], len(str(val)))
    
    # 打印表头
    header = ' | '.join(c.ljust(col_widths[i]) for i, c in enumerate(columns))
    sep = '-+-'.join('-' * col_widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for row in result:
        print(' | '.join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
    print(f'\n({len(result)} row{"s" if len(result) != 1 else ""})')


# ============================================
# 创建数据库（存在文件里，关闭 notebook 后数据还在）
# ============================================
conn = sqlite3.connect('books.db')
conn.execute("PRAGMA foreign_keys = ON")

# 如果表已存在就删掉重建（保证每次运行结果一致）
conn.execute("DROP TABLE IF EXISTS books;")

conn.execute('''
CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    year INTEGER,
    rating REAL,
    pages INTEGER,
    genre TEXT,
    publisher TEXT,
    votes INTEGER,
    translator TEXT
);
''')

# 样本数据（25 本畅销书）
books_data = [
    ('The Great Gatsby', 'F. Scott Fitzgerald', 1925, 3.93, 180, 'Fiction', 'Scribner', 4500000, None),
    ('To Kill a Mockingbird', 'Harper Lee', 1960, 4.27, 281, 'Fiction', 'J.B. Lippincott', 5200000, None),
    ('1984', 'George Orwell', 1949, 4.19, 328, 'Fiction', 'Secker & Warburg', 4100000, None),
    ('The Alchemist', 'Paulo Coelho', 1988, 3.91, 197, 'Fiction', 'HarperCollins', 2500000, 'Alan R. Clarke'),
    ('Educated', 'Tara Westover', 2018, 4.47, 334, 'Nonfiction', 'Random House', 1500000, None),
    ('Becoming', 'Michelle Obama', 2018, 4.53, 448, 'Nonfiction', 'Crown', 2100000, None),
    ('Sapiens', 'Yuval Noah Harari', 2011, 4.42, 443, 'Nonfiction', 'Harper', 1800000, 'Haim Watzman'),
    ('Gone Girl', 'Gillian Flynn', 2012, 4.13, 432, 'Mystery', 'Crown', 2200000, None),
    ('The Girl with the Dragon Tattoo', 'Stieg Larsson', 2005, 4.16, 465, 'Mystery', 'Norstedts', 3100000, 'Reg Keeland'),
    ('Dune', 'Frank Herbert', 1965, 4.26, 688, 'Science Fiction', 'Chilton', 1800000, None),
    ('Neuromancer', 'William Gibson', 1984, 3.89, 271, 'Science Fiction', 'Ace', 1100000, None),
    ('The Shining', 'Stephen King', 1977, 4.25, 447, 'Horror', 'Doubleday', 1600000, None),
    ('It', 'Stephen King', 1986, 4.25, 1138, 'Horror', 'Viking', 1300000, None),
    ('Atomic Habits', 'James Clear', 2018, 4.35, 320, 'Nonfiction', 'Avery', 2800000, None),
    ('The Martian', 'Andy Weir', 2011, 4.41, 369, 'Science Fiction', 'Crown', 1900000, None),
    ('Pride and Prejudice', 'Jane Austen', 1813, 4.28, 279, 'Fiction', 'T. Egerton', 3800000, None),
    ('The Catcher in the Rye', 'J.D. Salinger', 1951, 3.81, 214, 'Fiction', 'Little, Brown', 3300000, None),
    ('The Silent Patient', 'Alex Michaelides', 2019, 4.18, 325, 'Mystery', 'Celadon', 1200000, None),
    ('Project Hail Mary', 'Andy Weir', 2021, 4.52, 496, 'Science Fiction', 'Ballantine', 1800000, None),
    ('Thinking, Fast and Slow', 'Daniel Kahneman', 2011, 4.17, 499, 'Nonfiction', 'FSG', 1500000, None),
    ('The Road', 'Cormac McCarthy', 2006, 4.0, 287, 'Fiction', 'Knopf', 1100000, None),
    ('Where the Crawdads Sing', 'Delia Owens', 2018, 4.39, 368, 'Fiction', 'Putnam', 2400000, None),
    ('Circe', 'Madeline Miller', 2018, 4.27, 393, 'Fiction', 'Little, Brown', 1300000, None),
    ('A Brief History of Time', 'Stephen Hawking', 1988, 4.18, 256, 'Nonfiction', 'Bantam', 1900000, None),
    ("The Hitchhiker's Guide to the Galaxy", 'Douglas Adams', 1979, 4.22, 193, 'Science Fiction', 'Pan', 2200000, None),
]

conn.executemany(
    'INSERT INTO books (title, author, year, rating, pages, genre, publisher, votes, translator) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)',
    books_data
)
conn.commit()

# 看看表结构
print('📋 books 表结构：')
print('   Column          Type        Constraints')
print('   -------------   ---------   -----------')
for row in conn.execute("PRAGMA table_info('books')"):
    constraints = []
    if row[3]: constraints.append('NOT NULL')
    if row[5]: constraints.append('PRIMARY KEY')
    constr = ', '.join(constraints) if constraints else ''
    print(f'   {row[1]:15s}  {row[2]:10s}  {constr}')

print(f'\n✅ 数据库创建完毕，共 {conn.execute("SELECT COUNT(*) FROM books").fetchone()[0]} 本书')
print('💡 数据库文件：books.db（在 notebook 同目录下）')
print('\n现在往下翻，开始学 SQL ↓')

---
## 1.3 基本查询

### SELECT — 挑选列

```sql
SELECT 列1, 列2 FROM 表名;
SELECT * FROM 表名;      -- * 代表所有列
```

In [ ]:
# 查看全部数据
sql('SELECT * FROM books;')

In [ ]:
# 只查书名和作者
sql('SELECT title, author FROM books;')

### WHERE — 过滤行

```sql
SELECT 列 FROM 表 WHERE 条件;
```

**执行逻辑**：`FROM`（找表）→ `WHERE`（逐行检查，只留满足的）→ `SELECT`（挑列）

In [ ]:
# 查 2018 年出版的书
sql("SELECT title, author, year FROM books WHERE year = 2018;")

In [ ]:
# 查 Fiction 类型且评分 > 4.0 的
sql("SELECT title, rating, genre FROM books WHERE genre = 'Fiction' AND rating > 4.0;")

### LIMIT — 限制返回行数

```sql
SELECT ... LIMIT n;
```

In [ ]:
# 只看前 5 本
sql('SELECT title, author FROM books LIMIT 5;')

### ✏️ 自己动手

写一条 SQL，查询**页数少于 300** 的书，只显示书名、作者和页数。

In [ ]:
# 👇 在引号里写你的 SQL
sql("")


---
## 1.4 过滤与运算符

### 比较运算符

`=` `!=` / `<>` `>` `<` `>=` `<=`

In [ ]:
# 评分 > 4.3 的书
sql('SELECT title, rating FROM books WHERE rating > 4.3;')

In [ ]:
# 页数在 300 到 400 之间的（用 >= 和 <=）
sql('SELECT title, pages FROM books WHERE pages >= 300 AND pages <= 400;')

### 逻辑运算符

| 运算符 | 含义 | 效果 |
|--------|------|------|
| `AND` | 所有条件都满足 | 结果更少 |
| `OR` | 满足任一即可 | 结果更多 |
| `NOT` | 取反 | 排除 |

> ⚠️ **优先级**：`AND` > `OR`。拿不准就加括号，不要猜！

In [ ]:
# AND：评分高 且 页数少的书
sql('SELECT title, genre, rating, pages FROM books WHERE rating > 4.0 AND pages < 300;')

In [ ]:
# OR：Mystery 或 Horror 类型
sql("SELECT title, genre FROM books WHERE genre = 'Mystery' OR genre = 'Horror';")

In [ ]:
# 混合 AND/OR — 必须用括号明确意图！
# 不加括号的话，AND 先执行，逻辑就变了
sql("""SELECT title, genre, rating FROM books
WHERE (genre = 'Fiction' OR genre = 'Mystery') AND rating >= 4.0;""")

In [ ]:
# NOT：排除 Nonfiction
sql("SELECT title, genre FROM books WHERE NOT genre = 'Nonfiction';")

### LIKE — 模糊匹配

| 通配符 | 含义 | 例子 |
|--------|------|------|
| `%` | 0 个或多个任意字符 | `'The%'` → The xxx |
| `_` | 恰好 1 个任意字符 | `'__at'` → That, What, Brat |

In [ ]:
# 'The' 开头的书名
sql("SELECT title FROM books WHERE title LIKE 'The%';")

In [ ]:
# 书名中包含 'the'（不区分大小写... SQLite LIKE 默认不区分）
sql("SELECT title FROM books WHERE title LIKE '%the%';")

In [ ]:
# 第二个字母是 'h' 的书名（_ 匹配 1 个字符）
sql("SELECT title FROM books WHERE title LIKE '_h%';")

### NULL — 空值处理

`NULL` = 未知/缺失。**不是 0，不是空字符串，不等于任何值（包括它自己）。**

> 🚨 `NULL = NULL` → `NULL`（未知），不是 `TRUE`！

判断 NULL 必须用 `IS NULL` 或 `IS NOT NULL`。

In [ ]:
# ❌ 错误示范：永远返回空
print('WHERE translator = NULL → 永远空结果（因为 NULL = NULL 不是 TRUE）\n')
sql('SELECT title, translator FROM books WHERE translator = NULL;')

In [ ]:
# ✅ 正确：有译者的书
sql('SELECT title, translator FROM books WHERE translator IS NOT NULL;')

In [ ]:
# ✅ 正确：没有译者的书（原版英文书）
sql('SELECT title FROM books WHERE translator IS NULL;')

### ✏️ 自己动手

写一条 SQL，找出**书名以 'T' 开头，或者有译者信息**的书。（提示：用 `OR` 连接两个条件）

In [ ]:
# 👇 在这里写
sql("")


---
## 1.5 范围与模式匹配

### BETWEEN ... AND ...

等价于 `>= ... AND <= ...`，**包含边界**。

In [ ]:
# 页数在 200-400 之间的书
sql('SELECT title, pages FROM books WHERE pages BETWEEN 200 AND 400;')

In [ ]:
# BETWEEN 也可用于文本（字典序）
sql("SELECT title FROM books WHERE title BETWEEN 'A' AND 'D';")

### IN (...) — 列表匹配

替代多个 `OR`，更简洁。

In [ ]:
# 找出指定类型
sql("SELECT title, genre FROM books WHERE genre IN ('Mystery', 'Horror', 'Science Fiction');")

In [ ]:
# NOT IN：排除
sql("SELECT title, genre FROM books WHERE genre NOT IN ('Fiction', 'Nonfiction');")

### ✏️ 自己动手

找出 **Stephen King 写的、出版于 1975-1990 年之间** 的书。

In [ ]:
# 👇 在这里写
sql("")


---
## 1.6 排序（ORDER BY）

```sql
ORDER BY 列 ASC;    -- 升序（默认，可不写）
ORDER BY 列 DESC;   -- 降序
```

In [ ]:
# 按评分从高到低
sql('SELECT title, rating FROM books ORDER BY rating DESC;')

In [ ]:
# 按年份升序（最老的排前面）
sql('SELECT title, year FROM books ORDER BY year ASC;')

### 多列排序

先按第一列排，值相同的再按第二列排。

In [ ]:
# 同年份的书按评分降序排
sql('SELECT title, year, rating FROM books ORDER BY year DESC, rating DESC;')

In [ ]:
# 同作者的书按年份降序排（新书在前）
sql('SELECT author, title, year FROM books ORDER BY author ASC, year DESC;')

### ORDER BY + WHERE + LIMIT 组合

**逻辑执行顺序**：

```
FROM（找表）→ WHERE（过滤）→ SELECT（选列）→ ORDER BY（排序）→ LIMIT（截取）
```

In [ ]:
# 2010 年以来评分最高的 5 本书
sql('''SELECT title, rating, year FROM books
WHERE year >= 2010 AND rating IS NOT NULL
ORDER BY rating DESC
LIMIT 5;''')

### ✏️ 自己动手

找出**页数最多的 3 本书**（不要那本 1138 页的怪兽哦，应该就是它排第一）。

In [ ]:
# 👇 在这里写
sql("")


---
## 1.7 聚合函数

把多行汇总成一行。

| 函数 | 作用 | 忽略 NULL？ |
|------|------|------------|
| `COUNT(*)` | 数行数 | 不忽略 |
| `COUNT(col)` | 数某列非 NULL 行数 | 忽略 |
| `AVG(col)` | 平均值 | 忽略 |
| `MAX(col)` | 最大值 | 忽略 |
| `MIN(col)` | 最小值 | 忽略 |
| `SUM(col)` | 求和 | 忽略 |

In [ ]:
# COUNT：统计
sql('SELECT COUNT(*) AS total FROM books;')

In [ ]:
# COUNT(*) vs COUNT(col)：NULL 不算！
sql('''SELECT
    COUNT(*) AS total,
    COUNT(translator) AS has_translator,
    COUNT(*) - COUNT(translator) AS no_translator
FROM books;''')

In [ ]:
# AVG, MAX, MIN, SUM 全家桶
sql('''SELECT
    ROUND(AVG(rating), 2) AS avg_rating,
    MAX(rating) AS highest,
    MIN(rating) AS lowest,
    ROUND(AVG(pages)) AS avg_pages,
    SUM(votes) AS total_votes
FROM books;''')

### 聚合 + WHERE = 过滤后再聚合

In [ ]:
# 只看 2010 年之后出版的书
sql('''SELECT
    COUNT(*) AS count,
    ROUND(AVG(rating), 2) AS avg_rating,
    ROUND(AVG(pages)) AS avg_pages
FROM books
WHERE year >= 2010;''')

### ⚠️ 常见错误：WHERE 里不能放聚合函数

```sql
-- ❌ 错误！
SELECT * FROM books WHERE AVG(rating) > 4.0;

-- 为什么？因为 WHERE 执行在聚合之前！
-- FROM → WHERE → SELECT（聚合在这里）→ ...
-- WHERE 的时候还没有 AVG 的结果

-- ✅ 过滤聚合结果用 HAVING（后面会学）
```

### ✏️ 自己动手

统计 **Stephen King 写了几本书，平均评分是多少**。

In [ ]:
# 👇 在这里写
sql("")


---
## 🎯 综合挑战

用本章学到的全部知识，回答以下 5 个问题。

In [ ]:
# Q1：Fiction 类型中评分最高的 3 本书是什么？
sql("")


In [ ]:
# Q2：书名以 'The' 开头的、2000 年之后出版的书有几本？
sql("")


In [ ]:
# Q3：Science Fiction 类型的平均页数是多少？（用 ROUND 保留整数）
sql("")


In [ ]:
# Q4：列出所有有译者的书，按评分降序排列
sql("")


In [ ]:
# Q5：出版年份最早和最晚的书各是哪一年？相差多少年？
# （提示：一条 SQL，用 MAX 和 MIN）
sql("")


---
## ✅ 检查清单

对照一下，都掌握了吗？

- [ ] `SELECT` + `WHERE` + `LIMIT` 基本查询
- [ ] `AND` / `OR` / `NOT` 的区别和优先级
- [ ] `LIKE` + `%` / `_` 模糊匹配
- [ ] `NULL` 只能用 `IS NULL`，不能用 `=`
- [ ] `BETWEEN` 和 `IN` 简化范围查询
- [ ] `ORDER BY` 单列和多列排序
- [ ] `COUNT` / `AVG` / `MAX` / `MIN` / `SUM`
- [ ] NULL 在聚合中会被忽略
- [ ] `FROM → WHERE → SELECT → ORDER BY → LIMIT` 执行顺序

---

> 📖 下一章：[02 - 数据库设计](../02-database-design/) — 去重、ER 图、主键与外键